# Notebook 6: Grid Search & Hyperparameter Tuning

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Topic:** Regularization & Optimization — Week 8  
**Dataset:** Synthetic California-style house price data (500 samples, 20 features)

---

## Why This Matters

You now know that regularization strength α has a huge impact on model quality — too small and you overfit, too large and you underfit. But how do you actually *choose* the right α (or any other hyperparameter)?

**Grid search** is the systematic answer: define a candidate set of hyperparameter values, evaluate every combination using cross-validation, and let the data guide you to the best configuration. It is:

- **Transparent** — you can see the full score landscape
- **Reproducible** — same grid, same data → same answer
- **Guaranteed to find the best value *within your grid*** (not necessarily globally)

Understanding grid search — including its pitfalls — is essential before moving to smarter alternatives like random search or Bayesian optimization.

## Real-World Analogy: Perfecting the Sunday Roast

You want the perfect roast chicken. Two key decisions: **oven temperature** and **cooking time**.

**Grid search approach:** test every combination on your menu:

| | 1.5 hours | 2 hours | 2.5 hours | 3 hours |
|---|---|---|---|---|
| **160°C** | raw | ok | good | dry |
| **175°C** | ok | **perfect** | good | dry |
| **190°C** | good | good | overcooked | burnt |
| **205°C** | ok | overcooked | burnt | burnt |

You systematically cook 16 chickens (4 temps × 4 times), rate each one, and pick the winner. Thorough — but expensive (16 chickens!).

**In machine learning:** each chicken is a *model fit*. Your grid is the table of hyperparameter combinations. Each cell's rating is the cross-validated score. You want to find `175°C / 2 hours` — the optimal α without tasting the test-set chicken until the very end.

> **Key warning:** If you keep tweaking the grid after peeking at the test chicken, you're not finding the best general recipe — you're just memorizing *that one chicken*.

## Plain English: Parameters vs. Hyperparameters

| | **Model Parameters** | **Hyperparameters** |
|---|---|---|
| **What?** | Values learned from data | Values set *before* training |
| **Examples** | Weights β in linear regression | α (regularization), k (neighbors), max_depth |
| **How set?** | Optimization (gradient descent, OLS) | You choose them (grid search, etc.) |
| **Can gradient descent learn them?** | Yes | Not directly — they control the learning process itself |

### The Grid Search Procedure

1. **Split data** into train+validation (for CV) and a held-out test set
2. **Define grid** — discrete values for each hyperparameter
3. **For each combination:**
   - Run k-fold cross-validation on the training set
   - Record mean CV score (and standard deviation)
4. **Select best combination** by CV score
5. **Retrain** on full training set with best params
6. **Evaluate once** on the held-out test set — this is your final reported score

> **Golden rule:** The test set is touched **exactly once**. Using it to iterate grid ranges is data leakage.

## Section 1: Setup & Synthetic House Price Dataset

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from itertools import product

from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    cross_val_score, KFold, train_test_split, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Reproducibility and aesthetics
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print('Libraries loaded. Ready to build grid search from scratch!')

In [ ]:
# ── Build synthetic California-style house price dataset ─────────────────────
# 500 samples, 20 features (12 real / informative, 8 noise)

n_samples   = 500
n_real      = 12
n_noise     = 8
n_features  = n_real + n_noise

# Feature names
real_names  = ['sq_ft', 'bedrooms', 'bathrooms', 'lot_size', 'age_years',
               'garage', 'school_rating', 'distance_cbd', 'renovated',
               'pool', 'stories', 'neighborhood_score']
noise_names = [f'noise_{i}' for i in range(1, n_noise + 1)]
feat_names  = real_names + noise_names

# True coefficients
true_coef_real  = np.array([180, 25, 40, 35, -9, 30, 22, -28, 45, 38, 15, 20])
true_coef_noise = np.zeros(n_noise)
true_coef       = np.concatenate([true_coef_real, true_coef_noise])

# Generate feature matrix with some mild correlations
X_raw = np.random.randn(n_samples, n_features)
# Add correlation: bathrooms ~ bedrooms
X_raw[:, 2] = 0.7 * X_raw[:, 1] + 0.3 * np.random.randn(n_samples)

y_raw = X_raw @ true_coef + 300 + np.random.randn(n_samples) * 60

df = pd.DataFrame(X_raw, columns=feat_names)
df['price_kUSD'] = y_raw

# ─ Train / test split (hold out test set — do NOT touch until the very end) ─
X = df[feat_names].values
y = df['price_kUSD'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize on train, apply to test
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set:  {X_train_sc.shape[0]} samples')
print(f'Test set:      {X_test_sc.shape[0]} samples  (HELD OUT)')
print(f'Features:      {n_features} ({n_real} real, {n_noise} noise)')
print(f'Price range (train): ${y_train.min():.0f}K – ${y_train.max():.0f}K')

## Section 2: Grid Search from Scratch

Before using sklearn's `GridSearchCV`, let's build it ourselves so we understand exactly what it does.

In [ ]:
# ── grid_search_cv: a hand-rolled implementation ─────────────────────────────
def grid_search_cv(model_class, param_grid, X, y, cv=5, scoring='r2', verbose=False):
    """
    Exhaustive cross-validated hyperparameter search.

    Parameters
    ----------
    model_class : sklearn estimator class (not instance)
    param_grid  : dict mapping param name -> list of values
    X, y        : training data
    cv          : number of folds
    scoring     : 'r2' or 'neg_mean_squared_error'

    Returns
    -------
    best_params : dict
    best_score  : float
    results_df  : DataFrame with all combinations and their scores
    """
    keys   = list(param_grid.keys())
    values = list(param_grid.values())

    best_score  = -np.inf
    best_params = None
    results     = []

    all_combos = list(product(*values))   # cartesian product
    if verbose:
        print(f'Total combinations: {len(all_combos)} | CV folds: {cv} '
              f'→ {len(all_combos) * cv} model fits')

    for combo in all_combos:
        params = dict(zip(keys, combo))
        model  = model_class(**params, max_iter=10000)   # create fresh model

        # k-fold cross-validation on the given data
        cv_scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
        mean_score = cv_scores.mean()
        std_score  = cv_scores.std()

        results.append({**params, 'mean_cv_score': mean_score, 'std_cv_score': std_score})

        if mean_score > best_score:
            best_score  = mean_score
            best_params = params

    results_df = pd.DataFrame(results)
    return best_params, best_score, results_df


print('grid_search_cv defined. Key design decisions:')
print('  1. itertools.product generates all hyperparameter combinations')
print('  2. Fresh model instance created for each combination')
print('  3. cross_val_score handles k-fold splitting internally')
print('  4. We track mean AND std — std helps apply the 1-SE rule')

## Section 3: Run Grid Search on Ridge — 1D Grid

In [ ]:
# ── Grid search over Ridge alpha ──────────────────────────────────────────────
ridge_param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
}

t0 = time.time()
best_params_ridge, best_score_ridge, ridge_results = grid_search_cv(
    Ridge,
    ridge_param_grid,
    X_train_sc,
    y_train,
    cv=5,
    scoring='r2',
    verbose=True
)
t_ridge = time.time() - t0

print(f'\nTime taken: {t_ridge:.2f}s')
print(f'Best Ridge alpha: {best_params_ridge["alpha"]}')
print(f'Best CV R²: {best_score_ridge:.4f}')
print('\nFull results table:')
print(ridge_results.sort_values('mean_cv_score', ascending=False).to_string(index=False))

In [ ]:
# ── Plot Ridge CV score vs. alpha (1D grid) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

alphas_plot = ridge_results['alpha'].values
scores_plot = ridge_results['mean_cv_score'].values
stds_plot   = ridge_results['std_cv_score'].values

ax.semilogx(alphas_plot, scores_plot, 'o-', color='steelblue', lw=2.5, ms=8, label='Mean CV R²')
ax.fill_between(alphas_plot,
                scores_plot - stds_plot,
                scores_plot + stds_plot,
                alpha=0.25, color='steelblue', label='±1 std')
ax.axvline(best_params_ridge['alpha'], color='red', lw=2, ls=':',
           label=f'Best α = {best_params_ridge["alpha"]}')

# Annotate each point
for a, s in zip(alphas_plot, scores_plot):
    ax.text(a, s + 0.003, f'{s:.4f}', ha='center', fontsize=8, color='navy')

ax.set_xlabel('Ridge alpha (log scale)')
ax.set_ylabel('CV R² (higher = better)')
ax.set_title('Figure 1: Ridge Hyperparameter Search\nCV Score vs. Regularization Strength')
ax.legend()
plt.tight_layout()
plt.show()

print('Note: If the best alpha is at the boundary of your grid (first or last value),')
print('the true optimum may be OUTSIDE the grid. This is the "wrong range" pitfall.')

## Section 4: 2D Grid Search on ElasticNet — alpha × l1_ratio

In [ ]:
# ── 2D grid search: ElasticNet alpha × l1_ratio ───────────────────────────────
en_param_grid = {
    'alpha':    [0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

n_combos = len(en_param_grid['alpha']) * len(en_param_grid['l1_ratio'])
print(f'ElasticNet grid: {len(en_param_grid["alpha"])} alpha values × '
      f'{len(en_param_grid["l1_ratio"])} l1_ratio values = {n_combos} combinations')
print(f'With 5-fold CV: {n_combos * 5} total model fits')

t0 = time.time()
best_params_en, best_score_en, en_results = grid_search_cv(
    ElasticNet,
    en_param_grid,
    X_train_sc,
    y_train,
    cv=5,
    scoring='r2',
    verbose=True
)
t_en = time.time() - t0

print(f'\nTime taken: {t_en:.2f}s')
print(f'Best ElasticNet params: {best_params_en}')
print(f'Best CV R²: {best_score_en:.4f}')

In [ ]:
# ── Heatmap of CV R² scores: alpha × l1_ratio ────────────────────────────────
# Pivot the results into a 2D matrix for the heatmap
pivot_df = en_results.pivot_table(
    index='alpha', columns='l1_ratio', values='mean_cv_score'
)

fig, ax = plt.subplots(figsize=(10, 6))

hm = sns.heatmap(
    pivot_df,
    annot=True, fmt='.4f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'CV R²'}
)

# Find best cell and highlight it
best_alpha_val    = best_params_en['alpha']
best_l1_val       = best_params_en['l1_ratio']
best_row_idx      = list(pivot_df.index).index(best_alpha_val)
best_col_idx      = list(pivot_df.columns).index(best_l1_val)

ax.add_patch(plt.Rectangle(
    (best_col_idx, best_row_idx), 1, 1,
    fill=False, edgecolor='blue', lw=3, label='Best combination'
))

ax.set_title(f'Figure 2: ElasticNet Grid Search — CV R² Heatmap\n'
             f'Best: alpha={best_alpha_val}, l1_ratio={best_l1_val} (blue outline)')
ax.set_xlabel('l1_ratio  (0=Ridge, 1=Lasso)')
ax.set_ylabel('alpha  (regularization strength)')
plt.tight_layout()
plt.show()

print(f'Best cell: alpha={best_alpha_val}, l1_ratio={best_l1_val}, R²={best_score_en:.4f}')
print('Pattern: warmer colors = better scores. The gradient shows the effect of each hyperparameter.')

## Section 5: Compare Scratch Implementation vs. sklearn `GridSearchCV`

In [ ]:
# ── sklearn GridSearchCV: verify it gives the same best params ─────────────────
# Note: sklearn's param_grid uses slightly different key format for Pipeline,
# but for a plain model it's the same dictionary.

sk_param_grid = {
    'alpha':    en_param_grid['alpha'],
    'l1_ratio': en_param_grid['l1_ratio']
}

t0 = time.time()
sk_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    sk_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,       # use all CPU cores (parallel)
    refit=True       # refit best model on full training set
)
sk_grid.fit(X_train_sc, y_train)
t_sk = time.time() - t0

print(f'sklearn GridSearchCV time: {t_sk:.2f}s (uses n_jobs=-1 parallelism)')
print(f'Our scratch impl. time:    {t_en:.2f}s (single core)')

print(f'\nsklearn best params: {sk_grid.best_params_}')
print(f'Scratch  best params: {best_params_en}')

params_match = (sk_grid.best_params_['alpha'] == best_params_en['alpha'] and
                sk_grid.best_params_['l1_ratio'] == best_params_en['l1_ratio'])
print(f'\nBest params match: {params_match}')
print(f'sklearn best CV R²: {sk_grid.best_score_:.6f}')
print(f'Scratch  best CV R²: {best_score_en:.6f}')

In [ ]:
# ── Compare all scores side by side ──────────────────────────────────────────
# sklearn stores results in cv_results_; build a comparison DataFrame.

sk_results_df = pd.DataFrame(sk_grid.cv_results_)[[
    'param_alpha', 'param_l1_ratio',
    'mean_test_score', 'std_test_score'
]].rename(columns={
    'param_alpha': 'alpha',
    'param_l1_ratio': 'l1_ratio',
    'mean_test_score': 'sk_mean_r2',
    'std_test_score': 'sk_std_r2'
})

scratch_results_df = en_results.rename(columns={
    'mean_cv_score': 'scratch_mean_r2',
    'std_cv_score': 'scratch_std_r2'
})

comparison_df = pd.merge(
    sk_results_df.astype({'alpha': float, 'l1_ratio': float}),
    scratch_results_df,
    on=['alpha', 'l1_ratio']
)
comparison_df['diff'] = (comparison_df['sk_mean_r2'] - comparison_df['scratch_mean_r2']).abs()

print('Comparison of sklearn vs. scratch (first 10 rows):')
print(comparison_df[['alpha','l1_ratio','sk_mean_r2','scratch_mean_r2','diff']]
      .sort_values('diff', ascending=False).head(10).to_string(index=False))
print(f'\nMax absolute score difference: {comparison_df["diff"].max():.8f}')
print('(Tiny differences may arise from random fold assignments — use random_state to fix.)')

## Section 6: Timing — The Curse of Dimensionality in Grid Search

Grid search scales as **m^k** where m = values per hyperparameter, k = number of hyperparameters. Adding one more hyperparameter multiplies the number of fits by m.

In [ ]:
# ── Timing: how does grid search scale with grid size? ────────────────────────
# We test 5×5, 10×10, and 20×20 grids over alpha × l1_ratio.
# Total fits = grid_size^2 × 5_folds

grid_sizes  = [5, 10, 20]
timing_results = []

for gs in grid_sizes:
    alpha_vals   = np.logspace(-3, 2, gs).tolist()
    l1_vals      = np.linspace(0.1, 0.9, gs).tolist()
    n_fits       = gs * gs * 5  # 5-fold CV

    t0 = time.time()
    sk_g = GridSearchCV(
        ElasticNet(max_iter=5000),
        {'alpha': alpha_vals, 'l1_ratio': l1_vals},
        cv=5, scoring='r2', n_jobs=-1
    )
    sk_g.fit(X_train_sc, y_train)
    elapsed = time.time() - t0

    timing_results.append({'grid_size': f'{gs}×{gs}', 'n_fits': n_fits, 'time_s': elapsed})
    print(f'Grid {gs}×{gs}: {n_fits} fits → {elapsed:.2f}s')

timing_df = pd.DataFrame(timing_results)
print('\nTiming summary:')
print(timing_df.to_string(index=False))

In [ ]:
# ── Plot grid size vs. time ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: time vs. number of fits
ax1.plot(timing_df['n_fits'], timing_df['time_s'], 'o-', color='steelblue', lw=2.5, ms=9)
for _, row in timing_df.iterrows():
    ax1.text(row['n_fits'] + 5, row['time_s'], f"{row['grid_size']}\n{row['time_s']:.2f}s",
             fontsize=9, ha='left', va='center')
ax1.set_xlabel('Total model fits (grid_size² × 5 folds)')
ax1.set_ylabel('Wall-clock time (seconds)')
ax1.set_title('Grid Search Scaling:\nTime vs. Total Fits')

# Right: extrapolate to 3 hyperparameters
# If 2D 20×20 = 400 combos, then 3D 20×20×20 = 8000 combos
dim_label  = ['5×5 (2D)', '10×10 (2D)', '20×20 (2D)', '20×20×10 (3D)', '20×20×20 (3D)']
dim_fits   = [125, 500, 2000, 20000, 40000]

# Estimate 3D times by linear extrapolation from measured rate
time_per_fit = timing_df['time_s'].iloc[-1] / timing_df['n_fits'].iloc[-1]
dim_times    = [f * time_per_fit for f in dim_fits]

colors_bar = ['steelblue'] * 3 + ['darkorange'] * 2
bars = ax2.barh(dim_label, dim_times, color=colors_bar, edgecolor='black', lw=0.6)
for bar, t in zip(bars, dim_times):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{t:.1f}s', va='center', fontsize=9)
ax2.set_xlabel('Estimated time (seconds)')
ax2.set_title('Scaling to 3 Hyperparameters:\nEstimated Time (orange = 3D)')

plt.tight_layout()
plt.suptitle('Figure 3: Exponential Scaling of Grid Search', y=1.02, fontsize=13)
plt.show()

print(f'Rate: ~{time_per_fit*1000:.2f}ms per model fit')
print('Adding one more hyperparameter with 20 values multiplies time by 20!')

## Section 7: The Wrong-Range Pitfall

Grid search can only find the best value *within your specified grid*. If the true optimum lies outside the grid, you'll get a suboptimal result — and the CV curve will plateau at the boundary, tipping you off.

In [ ]:
# ── Wrong range pitfall demonstration ─────────────────────────────────────────
# True optimal alpha for Ridge on this dataset is around 1–10.
# We'll set up a grid that's entirely too large (alpha = 100 to 100000)
# and show that the CV score keeps improving toward the LEFT edge — 
# meaning the optimum is outside (to the left of) our grid.

wrong_alphas = [100, 300, 1000, 3000, 10000, 30000, 100000]
good_alphas  = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

def cv_scores_for_alphas(alphas, X, y, cv=5):
    """Return mean CV R² for each Ridge alpha."""
    means, stds = [], []
    for a in alphas:
        s = cross_val_score(Ridge(alpha=a), X, y, cv=cv, scoring='r2')
        means.append(s.mean())
        stds.append(s.std())
    return np.array(means), np.array(stds)

wrong_means, wrong_stds = cv_scores_for_alphas(wrong_alphas, X_train_sc, y_train)
good_means,  good_stds  = cv_scores_for_alphas(good_alphas,  X_train_sc, y_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ─ Wrong range ─
ax1.semilogx(wrong_alphas, wrong_means, 'o-', color='crimson', lw=2.5, ms=8)
ax1.fill_between(wrong_alphas, wrong_means - wrong_stds, wrong_means + wrong_stds,
                 alpha=0.25, color='crimson')
ax1.annotate('Score keeps rising →\noptimum is LEFT of grid!',
             xy=(wrong_alphas[0], wrong_means[0]),
             xytext=(wrong_alphas[1], wrong_means[0] - 0.05),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax1.set_xlabel('Alpha (log scale)')
ax1.set_ylabel('CV R²')
ax1.set_title('WRONG range: grid too large\n(optimum outside grid — monotone curve)')

# ─ Good range ─
best_idx = np.argmax(good_means)
ax2.semilogx(good_alphas, good_means, 'o-', color='seagreen', lw=2.5, ms=8)
ax2.fill_between(good_alphas, good_means - good_stds, good_means + good_stds,
                 alpha=0.25, color='seagreen')
ax2.axvline(good_alphas[best_idx], color='red', lw=2, ls=':',
            label=f'Best α = {good_alphas[best_idx]}')
ax2.set_xlabel('Alpha (log scale)')
ax2.set_ylabel('CV R²')
ax2.set_title('GOOD range: grid includes optimum\n(curve has a clear peak)')
ax2.legend()

plt.suptitle('Figure 4: Wrong Range Pitfall in Grid Search', fontsize=13)
plt.tight_layout()
plt.show()

print('Diagnostic tip: if CV score is monotone (always increasing or decreasing),')
print('your grid does not contain the optimum. Expand the range in that direction.')

## Section 8: Final Evaluation — Test Set (Once Only)

In [ ]:
# ── Final model evaluation on the held-out test set ───────────────────────────
# We use the best params found by sklearn GridSearchCV (which also refits on
# full training set via refit=True). We evaluate ONCE.

# GridSearchCV's best_estimator_ is already refit on X_train_sc
y_pred_test = sk_grid.best_estimator_.predict(X_test_sc)

test_r2  = r2_score(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)

print('=' * 50)
print('FINAL TEST SET EVALUATION (evaluated once)')
print('=' * 50)
print(f'Best ElasticNet params: {sk_grid.best_params_}')
print(f'CV R² (validation):  {sk_grid.best_score_:.4f}')
print(f'Test R²:             {test_r2:.4f}')
print(f'Test RMSE:           ${test_rmse:.1f}K')
print('=' * 50)

# ─ Residual plot ─
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(y_test, y_pred_test, alpha=0.55, color='steelblue', edgecolors='white', lw=0.3)
lims = [min(y_test.min(), y_pred_test.min()) - 20,
        max(y_test.max(), y_pred_test.max()) + 20]
ax1.plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
ax1.set_xlabel('Actual Price ($K)')
ax1.set_ylabel('Predicted Price ($K)')
ax1.set_title(f'Test Set: Actual vs. Predicted\nTest R² = {test_r2:.4f}')
ax1.legend()

residuals = y_test - y_pred_test
ax2.scatter(y_pred_test, residuals, alpha=0.55, color='darkorange', edgecolors='white', lw=0.3)
ax2.axhline(0, color='red', lw=2, ls='--')
ax2.set_xlabel('Predicted Price ($K)')
ax2.set_ylabel('Residual ($K)')
ax2.set_title(f'Residual Plot\nRMSE = ${test_rmse:.1f}K')

plt.suptitle('Figure 5: Final Model Performance on Held-Out Test Set', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary: putting it all together ─────────────────────────────────────────
# Model comparison: default vs. grid-searched ElasticNet

# Default ElasticNet (alpha=1.0, l1_ratio=0.5)
default_en = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
default_en.fit(X_train_sc, y_train)
default_pred  = default_en.predict(X_test_sc)
default_test_r2 = r2_score(y_test, default_pred)

# Best model from grid search
best_en = sk_grid.best_estimator_
best_pred    = best_en.predict(X_test_sc)
best_test_r2 = r2_score(y_test, best_pred)

comparison = pd.DataFrame({
    'Model': ['Default ElasticNet (α=1, l1=0.5)', f'Grid-Searched ElasticNet'],
    'Alpha': [1.0, sk_grid.best_params_['alpha']],
    'l1_ratio': [0.5, sk_grid.best_params_['l1_ratio']],
    'CV R²': [cross_val_score(default_en, X_train_sc, y_train, cv=5, scoring='r2').mean(),
              sk_grid.best_score_],
    'Test R²': [default_test_r2, best_test_r2]
})

print('Model comparison:')
print(comparison.to_string(index=False))
print(f'\nImprovement from grid search: '
      f'+{(best_test_r2 - default_test_r2)*100:.2f} percentage points in test R²')

## Section 9: Pitfalls Recap & When to Use Alternatives

In [ ]:
# ── Pitfalls recap with visual summary ───────────────────────────────────────

pitfalls = [
    ('Wrong range',
     'Grid does not contain the optimum',
     'CV curve is monotone → expand range'),
    ('Too coarse grid',
     'Optimum lies between two tested values',
     'Use finer grid around best value (zoom-in)'),
    ('Exponential scaling',
     'm^k combinations for k hyperparams, m values each',
     'Use RandomSearchCV or Bayesian Optimization'),
    ('Test set leakage',
     'Checking test score to adjust grid → data leakage',
     'Touch test set ONCE after all tuning is done'),
    ('Single CV run',
     'Randomness in CV folds affects best params',
     'Use repeated CV (RepeatedKFold) for stability'),
]

print('Grid Search Pitfalls & Solutions:')
print('=' * 80)
for i, (pitfall, description, solution) in enumerate(pitfalls, 1):
    print(f'{i}. PITFALL:  {pitfall}')
    print(f'   Problem:  {description}')
    print(f'   Solution: {solution}')
    print()

print('=' * 80)
print('\nAlternatives to Grid Search (for large hyperparameter spaces):')
alts = [
    ('RandomizedSearchCV', 'Sample random combos — often finds good params 10× faster'),
    ('Bayesian Optimization', 'Uses past results to choose next point intelligently'),
    ('Hyperband / ASHA', 'Early-stops bad configs — great for neural networks'),
    ('Optuna / Ray Tune', 'Production-grade libraries for large-scale HPO'),
]
for name, note in alts:
    print(f'  • {name:30s}: {note}')

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You grid-search 5 hyperparameters with 10 values each using 5-fold cross-validation. How many model fits does this require?

<details>
<summary>Show Answer</summary>

**500,000 model fits.**

Calculation:
- Total hyperparameter combinations = 10⁵ = 100,000
- Each combination requires 5 separate model fits (one per fold)
- Total fits = 100,000 × 5 = **500,000**

This is the core problem with grid search at scale. If each fit takes 1 second, that's 5.8 days of computation. For 5 hyperparameters with 10 values, RandomizedSearchCV with just 200 combinations (2,000 fits × 5 folds = 10,000 fits total) often finds a result within 5% of the grid search optimum — in 1/50th the time.

</details>

---

**Question 2:** After grid search selects the best params, you evaluate on the test set and report that score. Later you tune the grid and do it again. Why is the test score now unreliable?

<details>
<summary>Show Answer</summary>

**You have leaked information from the test set into your model selection process.**

When you see the test score, adjust your grid based on it, and re-evaluate, you are effectively using the test set as another validation set. The test score is no longer an unbiased estimate of how your model will perform on truly unseen data because:

1. You selected the grid range *because* it performed well on the test data
2. The test score is now optimistically biased — it's the result of implicit optimization on that specific test set
3. If deployed on new data, performance will likely be lower than reported

**The fix:** Use a three-way split (train / validation / test) where the validation set is used for all tuning and the test set is used exactly once, at the very end, to report the final performance.

</details>

---

**Question 3:** Grid search on `[0.001, 0.01, 0.1, 1.0, 10]` for α finds the best value is α=10 (the largest in the grid). What should you do next?

<details>
<summary>Show Answer</summary>

**The optimum is likely outside your grid (too small values tested). Expand the grid to larger α values.**

When the best value lands at the boundary of your search range, it's a strong signal that the true optimum lies *beyond* that boundary. The CV curve is monotone in that direction.

Recommended next steps:
1. **Diagnose:** Plot CV score vs. α — if it's still rising at α=10, you need larger values.
2. **Expand range:** Try `[1, 10, 100, 1000, 10000]` — always overlap with your current best to confirm the interior peak.
3. **Zoom in:** Once you find the region (e.g., between 100 and 1000), do a finer search in that range.
4. **Check both directions:** Also confirm the score drops below α=0.001, confirming the lower boundary is fine.

A safe initial strategy: always start with a very wide log-spaced grid (`np.logspace(-5, 5, 11)`) to locate the rough region, then zoom in.

</details>